1. Kaggle login

In [10]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


2. Download dataset from Kaggle

In [11]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

butterma_tensorforge_tickets_path = kagglehub.dataset_download('butterma/tensorforge-tickets')

print('Data source import complete.')


Data source import complete.


3. Mount Google Drive

In [12]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


4. Load and clean data

In [13]:
import unicodedata
import pandas as pd
from pathlib import Path

INPUT_ROOT = Path("/root/.cache/kagglehub/datasets/butterma/tensorforge-tickets")
TARGET_COLUMNS = ["category", "secondary_category", "is_urgent"]


def find_data_file(file_name):
    matches = list(INPUT_ROOT.rglob(file_name))
    if not matches:
        raise FileNotFoundError(f"{file_name} not found under {INPUT_ROOT}. Is the dataset downloaded?")
    print("using", matches[0])
    return matches[0]


def convert_urgent_to_boolean(dataframe):
    dataframe["is_urgent"] = dataframe["is_urgent"].astype(str).str.strip().str.lower() == "true"
    return dataframe


def convert_empty_secondary_to_none(dataframe):
    secondary = dataframe["secondary_category"].astype(str).str.strip()
    dataframe["secondary_category"] = secondary.replace({"": "none", "nan": "none"})
    return dataframe


def load_split(file_name):
    file_path = find_data_file(file_name)
    dataframe = pd.read_csv(file_path, encoding="utf-8", dtype=str, keep_default_na=False)
    dataframe = convert_urgent_to_boolean(dataframe)
    return convert_empty_secondary_to_none(dataframe)


def normalize_text(raw_text):
    if raw_text is None or pd.isna(raw_text):
        return ""
    normalized = unicodedata.normalize("NFC", str(raw_text))
    return " ".join(normalized.split()).lower()


def build_model_input(row):
    subject = normalize_text(row["subject"])
    body = normalize_text(row["text"])
    return f"[{row['channel']}] {subject} {body}".strip()


def add_model_input_column(dataframe):
    dataframe["model_input"] = dataframe.apply(build_model_input, axis=1)
    return dataframe


train_data = add_model_input_column(load_split("train.csv"))
validation_data = add_model_input_column(load_split("validation.csv"))
print(train_data.shape, validation_data.shape)

using /root/.cache/kagglehub/datasets/butterma/tensorforge-tickets/versions/1/train.csv
using /root/.cache/kagglehub/datasets/butterma/tensorforge-tickets/versions/1/validation.csv
(4000, 9) (800, 9)


5. Train language detector + add language tags

In [ ]:
import joblib
import sklearn
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

DETECTOR_PATH = Path("/content/drive/MyDrive/tensorforge/language_detector.joblib")
print("sklearn", sklearn.__version__)


def build_detector_input(dataframe):
    subjects = dataframe["subject"].map(normalize_text)
    bodies = dataframe["text"].map(normalize_text)
    return (subjects + " " + bodies).str.strip()


def train_language_detector(train_data):
    detector = Pipeline([
        ("features", TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 4), min_df=2, sublinear_tf=True)),
        ("classifier", LinearSVC(C=1.0, class_weight="balanced")),
    ])
    detector.fit(build_detector_input(train_data), train_data["language"])
    return detector


def show_detector_quality(detector, validation_data):
    predicted_languages = detector.predict(build_detector_input(validation_data))
    print("language accuracy:", round(accuracy_score(validation_data["language"], predicted_languages), 4))
    print(classification_report(validation_data["language"], predicted_languages, digits=3))
    return predicted_languages


def build_tagged_input(subject, text, channel, language):
    return f"[{language}] [{channel}] {normalize_text(subject)} {normalize_text(text)}".strip()


def add_tagged_inputs(dataframe, languages):
    dataframe["model_input"] = [
        build_tagged_input(subject, text, channel, language)
        for subject, text, channel, language
        in zip(dataframe["subject"], dataframe["text"], dataframe["channel"], languages)
    ]
    return dataframe


language_detector = train_language_detector(train_data)
predicted_validation_languages = show_detector_quality(language_detector, validation_data)
train_data = add_tagged_inputs(train_data, train_data["language"])
validation_data = add_tagged_inputs(validation_data, predicted_validation_languages)
joblib.dump(language_detector, DETECTOR_PATH)
print(train_data["model_input"].iloc[0][:60])
print(validation_data["model_input"].iloc[0][:60])

sklearn 1.6.1
language accuracy: 0.9988
              precision    recall  f1-score   support

          en      1.000     1.000     1.000       280
       mixed      1.000     0.950     0.974        20
          si      1.000     1.000     1.000       160
    singlish      1.000     1.000     1.000       200
          ta      0.992     1.000     0.996       120
    tanglish      1.000     1.000     1.000        20

    accuracy                          0.999       800
   macro avg      0.999     0.992     0.995       800
weighted avg      0.999     0.999     0.999       800

[en] [chat]  i selected the service earlier today. i need he
[si] [email] විස්තර පරීක්ෂා කරන්න මට මේක බලලා දෙන්න පුළුවන්ද


6. Check language tags

In [ ]:
print(train_data["model_input"].iloc[0][:60])
print(validation_data["model_input"].iloc[0][:60])

[en] [chat]  i selected the service earlier today. i need he
[si] [email] විස්තර පරීක්ෂා කරන්න මට මේක බලලා දෙන්න පුළුවන්ද


7. Fine-tune xlm-roberta: CATEGORY model

In [ ]:
import numpy as np
import math
import torch
import transformers
from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import f1_score, accuracy_score

MODEL_NAME = "xlm-roberta-base"
OUTPUT_DIR = Path("/content/drive/MyDrive/tensorforge/xlmr_category")
MAX_TOKENS = 384

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("saving to", OUTPUT_DIR)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

def build_label_maps(labels):
    label_names = sorted(set(labels))
    label_to_id = {name: index for index, name in enumerate(label_names)}
    id_to_label = {index: name for name, index in label_to_id.items()}
    return label_to_id, id_to_label


def build_dataset(dataframe, label_to_id):
    return Dataset.from_dict({
        "text": dataframe["model_input"].tolist(),
        "label": [label_to_id[label] for label in dataframe["category"]],
    })


def tokenize_dataset(dataset, tokenizer):
    def tokenize_batch(batch):
        return tokenizer(batch["text"], truncation=True, max_length=MAX_TOKENS)
    return dataset.map(tokenize_batch, batched=True, remove_columns=["text"])


def compute_metrics(evaluation_output):
    logits, true_labels = evaluation_output
    predicted_labels = np.argmax(logits, axis=-1)
    return {
        "macro_f1": f1_score(true_labels, predicted_labels, average="macro"),
        "accuracy": accuracy_score(true_labels, predicted_labels),
    }


def count_warmup_steps(train_row_count, batch_size, epoch_count, warmup_fraction=0.1):
    steps_per_epoch = math.ceil(train_row_count / batch_size)
    return int(steps_per_epoch * epoch_count * warmup_fraction)


def build_training_arguments():
    batch_size = 16
    epoch_count = 6
    return TrainingArguments(
        output_dir=str(OUTPUT_DIR / "checkpoints"),
        num_train_epochs=epoch_count,
        learning_rate=2e-5,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=32,
        warmup_steps=count_warmup_steps(len(train_data), batch_size, epoch_count),
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        save_total_limit=1,
        fp16=torch.cuda.is_available(),
        report_to="none",
    )

def build_trainer(model, tokenizer, train_dataset, validation_dataset):
    return Trainer(
        model=model,
        args=build_training_arguments(),
        train_dataset=train_dataset,
        eval_dataset=validation_dataset,
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics,
    )


def score_by_language(trainer, validation_dataset, id_to_label):
    logits = trainer.predict(validation_dataset).predictions
    predicted_categories = np.array([id_to_label[index] for index in np.argmax(logits, axis=-1)])
    true_categories = validation_data["category"].to_numpy()
    languages = validation_data["language"].to_numpy()
    for language in sorted(set(languages)):
        language_mask = languages == language
        language_f1 = f1_score(true_categories[language_mask], predicted_categories[language_mask], average="macro")
        print(language, round(language_f1, 4))


def save_final_model(trainer, tokenizer):
    final_dir = OUTPUT_DIR / "final"
    trainer.save_model(str(final_dir))
    tokenizer.save_pretrained(str(final_dir))
    print("saved to", final_dir)
    print("torch", torch.__version__, "transformers", transformers.__version__)


label_to_id, id_to_label = build_label_maps(train_data["category"])
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(label_to_id), id2label=id_to_label, label2id=label_to_id
)

train_dataset = tokenize_dataset(build_dataset(train_data, label_to_id), tokenizer)
validation_dataset = tokenize_dataset(build_dataset(validation_data, label_to_id), tokenizer)

trainer = build_trainer(model, tokenizer, train_dataset, validation_dataset)
trainer.train()
print(trainer.evaluate())
score_by_language(trainer, validation_dataset, id_to_label)
save_final_model(trainer, tokenizer)

saving to /content/drive/MyDrive/tensorforge/xlmr_category
GPU: Tesla T4


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

...

Training Loss,Validation Loss,Epoch,Macro F1,Accuracy
0.004284,1.099547,6,0.810422,0.827500


{'eval_loss': 1.099547266960144, 'eval_macro_f1': 0.8104217123670788, 'eval_accuracy': 0.8275}


en 0.8467
mixed 0.774
si 0.6818
singlish 0.7331
ta 0.8006
tanglish 0.8701


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved to /content/drive/MyDrive/tensorforge/xlmr_category/final
torch 2.11.0+cu130 transformers 5.18.0


8. Calibrate category confidence + export (fp16)

In [ ]:
import json
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

FINAL_DIR = OUTPUT_DIR / "final"
EXPORT_DIR = OUTPUT_DIR / "xlmr_category"


def collect_validation_logits(trainer, validation_dataset):
    prediction_output = trainer.predict(validation_dataset)
    logits = torch.tensor(prediction_output.predictions, dtype=torch.float32)
    labels = torch.tensor(prediction_output.label_ids, dtype=torch.long)
    return logits, labels


def fit_temperature(logits, labels):
    log_temperature = torch.zeros(1, requires_grad=True)
    optimizer = torch.optim.LBFGS([log_temperature], lr=0.05, max_iter=200)
    loss_function = torch.nn.CrossEntropyLoss()

    def compute_loss():
        optimizer.zero_grad()
        loss = loss_function(logits / log_temperature.exp(), labels)
        loss.backward()
        return loss

    optimizer.step(compute_loss)
    return float(log_temperature.exp().item())


def expected_calibration_error(logits, labels, temperature, bin_count=10):
    probabilities = torch.softmax(logits / temperature, dim=-1)
    confidences, predictions = probabilities.max(dim=-1)
    correct = (predictions == labels).float()
    total_error = 0.0
    bin_edges = torch.linspace(0, 1, bin_count + 1)
    for lower, upper in zip(bin_edges[:-1], bin_edges[1:]):
        in_bin = (confidences > lower) & (confidences <= upper)
        if in_bin.any():
            gap = abs(confidences[in_bin].mean().item() - correct[in_bin].mean().item())
            total_error += in_bin.float().mean().item() * gap
    return round(total_error, 4)


def export_half_precision(final_dir, export_dir, temperature):
    model = AutoModelForSequenceClassification.from_pretrained(str(final_dir))
    model.half().save_pretrained(str(export_dir))
    AutoTokenizer.from_pretrained(str(final_dir)).save_pretrained(str(export_dir))
    with open(export_dir / "calibration.json", "w") as calibration_file:
        json.dump({"temperature": temperature}, calibration_file)
    print("exported to", export_dir)


validation_logits, validation_labels = collect_validation_logits(trainer, validation_dataset)
temperature = fit_temperature(validation_logits, validation_labels)
print("temperature:", round(temperature, 3))
print("ECE before:", expected_calibration_error(validation_logits, validation_labels, 1.0))
print("ECE after:", expected_calibration_error(validation_logits, validation_labels, temperature))
export_half_precision(FINAL_DIR, EXPORT_DIR, temperature)

temperature: 1.968
ECE before: 0.1498
ECE after: 0.0483


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

exported to /content/drive/MyDrive/tensorforge/xlmr_category/xlmr_category


9. Zip and download category model

In [ ]:
import shutil
from google.colab import files

zip_path = shutil.make_archive("/content/xlmr_category", "zip", OUTPUT_DIR / "xlmr_category")
print("zipped:", zip_path)
files.download(zip_path)

zipped: /content/xlmr_category.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

10. Fine-tune xlm-roberta: URGENCY model - EX 1

In [7]:
import hashlib
import json
import math
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import sklearn
import torch
import transformers
from datasets import Dataset
from sklearn.metrics import f1_score, precision_score, recall_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

MODEL_NAME = "xlm-roberta-base"
TENSORFORGE_DIR = Path("/content/drive/MyDrive/tensorforge")
DETECTOR_PATH = TENSORFORGE_DIR / "language_detector.joblib"
RUN_NAME = datetime.now().strftime("urgent_%Y%m%d_%H%M")
RUN_DIR = TENSORFORGE_DIR / "runs" / RUN_NAME
CHECKPOINT_DIR = Path("/content/checkpoints_urgent")
MAX_TOKENS = 384
BATCH_SIZE = 16
EPOCH_COUNT = 5
LEARNING_RATE = 2e-5
SCRATCH_URGENT_F1 = 0.8947
URGENT_LABELS = {0: "not_urgent", 1: "urgent"}


def build_detector_inputs(dataframe):
    return [
        f"{normalize_text(subject)} {normalize_text(text)}".strip()
        for subject, text in zip(dataframe["subject"], dataframe["text"])
    ]


def build_tagged_inputs(dataframe, languages):
    return [
        f"[{language}] [{channel}] {normalize_text(subject)} {normalize_text(text)}".strip()
        for subject, text, channel, language
        in zip(dataframe["subject"], dataframe["text"], dataframe["channel"], languages)
    ]


def build_urgent_dataset(texts, urgent_flags, tokenizer):
    dataset = Dataset.from_dict({"text": list(texts), "label": [int(flag) for flag in urgent_flags]})
    return dataset.map(
        lambda batch: tokenizer(batch["text"], truncation=True, max_length=MAX_TOKENS),
        batched=True,
        remove_columns=["text"],
    )


def urgent_probabilities(logits):
    return torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1)[:, 1].numpy()


def find_best_threshold(probabilities, true_labels):
    candidate_thresholds = np.arange(0.05, 0.96, 0.01)
    scores = [f1_score(true_labels, probabilities >= threshold, zero_division=0) for threshold in candidate_thresholds]
    best_index = int(np.argmax(scores))
    return round(float(candidate_thresholds[best_index]), 2), float(scores[best_index])


def compute_metrics(evaluation_output):
    logits, true_labels = evaluation_output
    probabilities = urgent_probabilities(logits)
    best_threshold, best_f1 = find_best_threshold(probabilities, true_labels)
    return {
        "urgent_f1": best_f1,
        "urgent_f1_at_half": f1_score(true_labels, probabilities >= 0.5, zero_division=0),
        "best_threshold": best_threshold,
    }


def build_training_arguments(train_row_count):
    total_steps = math.ceil(train_row_count / BATCH_SIZE) * EPOCH_COUNT
    return TrainingArguments(
        output_dir=str(CHECKPOINT_DIR),
        num_train_epochs=EPOCH_COUNT,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=32,
        warmup_steps=int(total_steps * 0.1),
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="urgent_f1",
        save_total_limit=1,
        fp16=torch.cuda.is_available(),
        report_to="none",
        seed=42,
    )


def report_by_language(probabilities, threshold):
    predicted = probabilities >= threshold
    true_flags = validation_data["is_urgent"].to_numpy().astype(bool)
    languages = validation_data["language"].to_numpy()
    for language in sorted(set(languages)):
        mask = languages == language
        print(f"  {language:9} urgent F1 {f1_score(true_flags[mask], predicted[mask], zero_division=0):.4f}"
              f"   ({int(true_flags[mask].sum())} urgent tickets)")


def hash_file(file_path):
    return hashlib.md5(Path(file_path).read_bytes()).hexdigest()


def export_run(trainer, tokenizer, threshold, scores):
    export_dir = RUN_DIR / "xlmr_urgent"
    export_dir.mkdir(parents=True, exist_ok=True)
    trainer.model.half().save_pretrained(str(export_dir))
    tokenizer.save_pretrained(str(export_dir))
    (export_dir / "urgency.json").write_text(json.dumps({"threshold": threshold}))
    run_info = {
        "run_name": RUN_NAME,
        "base_model": MODEL_NAME,
        "target": "is_urgent",
        "input_format": "[language] [channel] subject text, language from language_detector.joblib",
        "hyperparameters": {"epochs": EPOCH_COUNT, "learning_rate": LEARNING_RATE, "batch_size": BATCH_SIZE, "max_tokens": MAX_TOKENS, "seed": 42},
        "validation": scores,
        "train_csv_md5": hash_file(find_data_file("train.csv")),
        "validation_csv_md5": hash_file(find_data_file("validation.csv")),
        "versions": {"torch": torch.__version__, "transformers": transformers.__version__, "sklearn": sklearn.__version__},
    }
    (RUN_DIR / "run_info.json").write_text(json.dumps(run_info, indent=2))
    print("saved run to", RUN_DIR)


language_detector = joblib.load(DETECTOR_PATH)
detected_validation_languages = language_detector.predict(build_detector_inputs(validation_data))
train_texts = build_tagged_inputs(train_data, train_data["language"])
validation_texts = build_tagged_inputs(validation_data, detected_validation_languages)
print(train_texts[0][:60])
print(validation_texts[0][:60])

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label=URGENT_LABELS, label2id={name: index for index, name in URGENT_LABELS.items()}
)
train_dataset = build_urgent_dataset(train_texts, train_data["is_urgent"], tokenizer)
validation_dataset = build_urgent_dataset(validation_texts, validation_data["is_urgent"], tokenizer)

trainer = Trainer(
    model=model,
    args=build_training_arguments(len(train_data)),
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)
trainer.train()

validation_probabilities = urgent_probabilities(trainer.predict(validation_dataset).predictions)
true_flags = validation_data["is_urgent"].to_numpy().astype(int)
final_threshold, final_f1 = find_best_threshold(validation_probabilities, true_flags)
final_predictions = validation_probabilities >= final_threshold
final_scores = {
    "urgent_f1": round(final_f1, 4),
    "precision": round(precision_score(true_flags, final_predictions), 4),
    "recall": round(recall_score(true_flags, final_predictions), 4),
    "threshold": final_threshold,
    "scratch_model_urgent_f1": SCRATCH_URGENT_F1,
}
print(json.dumps(final_scores, indent=2))
report_by_language(validation_probabilities, final_threshold)
export_run(trainer, tokenizer, final_threshold, final_scores)

[en] [chat]  i selected the service earlier today. i need he
[si] [email] විස්තර පරීක්ෂා කරන්න මට මේක බලලා දෙන්න පුළුවන්ද


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "urgent_f1": 0.8143,
  "precision": 0.95,
  "recall": 0.7125,
  "threshold": 0.27,
  "scratch_model_urgent_f1": 0.8947
}
  en        urgent F1 0.7600   (29 urgent tickets)
  mixed     urgent F1 1.0000   (3 urgent tickets)
  si        urgent F1 0.8667   (17 urgent tickets)
  singlish  urgent F1 0.6429   (18 urgent tickets)
  ta        urgent F1 1.0000   (12 urgent tickets)
  tanglish  urgent F1 1.0000   (1 urgent tickets)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

using /root/.cache/kagglehub/datasets/butterma/tensorforge-tickets/versions/1/train.csv
using /root/.cache/kagglehub/datasets/butterma/tensorforge-tickets/versions/1/validation.csv
saved run to /content/drive/MyDrive/tensorforge/runs/urgent_20261010_0434


In [8]:
import shutil
from google.colab import files

zip_path = shutil.make_archive("/content/xlmr_urgent", "zip", RUN_DIR / "xlmr_urgent")
files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Fine-tune xlm-roberta: URGENCY model - EX 2

*Upload Sractch Modal*

In [15]:
from pathlib import Path
from google.colab import files

uploaded = files.upload()
scratch_folder = Path("/content/drive/MyDrive/tensorforge/scratch")
scratch_folder.mkdir(parents=True, exist_ok=True)
for file_name, file_bytes in uploaded.items():
    (scratch_folder / file_name).write_bytes(file_bytes)
    print("saved", scratch_folder / file_name)

Saving filler_sentences.joblib to filler_sentences.joblib
Saving is_urgent_model.joblib to is_urgent_model.joblib
saved /content/drive/MyDrive/tensorforge/scratch/filler_sentences.joblib
saved /content/drive/MyDrive/tensorforge/scratch/is_urgent_model.joblib


In [16]:
import json
import math
import re
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import sklearn
import torch
import transformers
from datasets import Dataset
from sklearn.metrics import f1_score, precision_score, recall_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

TENSORFORGE_DIR = Path("/content/drive/MyDrive/tensorforge")
CATEGORY_MODEL_DIR = TENSORFORGE_DIR / "xlmr_category" / "final"
SCRATCH_DIR = TENSORFORGE_DIR / "scratch"
DETECTOR_PATH = TENSORFORGE_DIR / "language_detector.joblib"
RUN_NAME = datetime.now().strftime("urgent_v2_%Y%m%d_%H%M")
RUN_DIR = TENSORFORGE_DIR / "runs" / RUN_NAME
CHECKPOINT_DIR = Path("/content/checkpoints_urgent_v2")
MAX_TOKENS = 384
BATCH_SIZE = 16
EPOCH_COUNT = 4
LEARNING_RATE = 1e-5
URGENT_LABELS = {0: "not_urgent", 1: "urgent"}

REFERENCE_PATTERN = re.compile(r"\b[a-z]{3,5}-[a-z0-9]{5,}\b")
AMOUNT_PATTERN = re.compile(r"(?:lkr|rs\.?)\s*[\d,.]+k?|[\d,.]*\d[\d,.]*k?(?:\s*(?:lkr|rupees?))?")
SENTENCE_SPLIT_PATTERN = re.compile(r"(?<=[.?!])\s+|\s+(?=agent:|customer:)")


def mask_entities(text):
    masked = REFERENCE_PATTERN.sub(" <ref> ", text)
    masked = AMOUNT_PATTERN.sub(" <amount> ", masked)
    return " ".join(masked.split())


def split_sentences(text):
    return [sentence.strip() for sentence in SENTENCE_SPLIT_PATTERN.split(text) if sentence.strip()]


def remove_filler(body, filler_sentences):
    kept_sentences = [sentence for sentence in split_sentences(body) if sentence not in filler_sentences]
    return " ".join(kept_sentences) if kept_sentences else body


def build_clean_body(subject, text, filler_sentences):
    body = mask_entities(f"{normalize_text(subject)} {normalize_text(text)}".strip())
    return remove_filler(body, filler_sentences)


def build_scratch_inputs(dataframe, filler_sentences):
    return [
        f"[{channel}] {build_clean_body(subject, text, filler_sentences)}"
        for subject, text, channel in zip(dataframe["subject"], dataframe["text"], dataframe["channel"])
    ]


def build_transformer_inputs(dataframe, languages, filler_sentences):
    return [
        f"[{language}] [{channel}] {build_clean_body(subject, text, filler_sentences)}"
        for subject, text, channel, language
        in zip(dataframe["subject"], dataframe["text"], dataframe["channel"], languages)
    ]


def build_detector_inputs(dataframe):
    return [
        f"{normalize_text(subject)} {normalize_text(text)}".strip()
        for subject, text in zip(dataframe["subject"], dataframe["text"])
    ]


def build_urgent_dataset(texts, urgent_flags, tokenizer):
    dataset = Dataset.from_dict({"text": list(texts), "label": [int(flag) for flag in urgent_flags]})
    return dataset.map(
        lambda batch: tokenizer(batch["text"], truncation=True, max_length=MAX_TOKENS),
        batched=True,
        remove_columns=["text"],
    )


def urgent_probabilities(logits):
    return torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1)[:, 1].numpy()


def find_best_threshold(scores, true_labels):
    candidate_thresholds = np.arange(0.05, 0.96, 0.01)
    f1_values = [f1_score(true_labels, scores >= threshold, zero_division=0) for threshold in candidate_thresholds]
    best_index = int(np.argmax(f1_values))
    return round(float(candidate_thresholds[best_index]), 2), float(f1_values[best_index])


def compute_metrics(evaluation_output):
    logits, true_labels = evaluation_output
    probabilities = urgent_probabilities(logits)
    best_threshold, best_f1 = find_best_threshold(probabilities, true_labels)
    return {"urgent_f1": best_f1, "best_threshold": best_threshold}


def build_training_arguments(train_row_count):
    total_steps = math.ceil(train_row_count / BATCH_SIZE) * EPOCH_COUNT
    return TrainingArguments(
        output_dir=str(CHECKPOINT_DIR),
        num_train_epochs=EPOCH_COUNT,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=32,
        warmup_steps=int(total_steps * 0.1),
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="urgent_f1",
        save_total_limit=1,
        fp16=torch.cuda.is_available(),
        report_to="none",
        seed=42,
    )


def load_model_from_category_checkpoint():
    return AutoModelForSequenceClassification.from_pretrained(
        str(CATEGORY_MODEL_DIR),
        num_labels=2,
        id2label=URGENT_LABELS,
        label2id={name: index for index, name in URGENT_LABELS.items()},
        ignore_mismatched_sizes=True,
    )


def sigmoid(values):
    return 1.0 / (1.0 + np.exp(-values))


def score_option(option_name, predicted_flags, true_flags, threshold=None):
    return {
        "option": option_name,
        "urgent_f1": round(f1_score(true_flags, predicted_flags, zero_division=0), 4),
        "precision": round(precision_score(true_flags, predicted_flags, zero_division=0), 4),
        "recall": round(recall_score(true_flags, predicted_flags, zero_division=0), 4),
        "threshold": threshold,
    }


def compare_options(transformer_probabilities, scratch_scores, true_flags):
    transformer_threshold, _ = find_best_threshold(transformer_probabilities, true_flags)
    blended_scores = 0.5 * transformer_probabilities + 0.5 * sigmoid(scratch_scores)
    blend_threshold, _ = find_best_threshold(blended_scores, true_flags)
    transformer_flags = transformer_probabilities >= transformer_threshold
    scratch_flags = scratch_scores > 0
    options = [
        score_option("A  transformer v2 alone", transformer_flags, true_flags, transformer_threshold),
        score_option("B  scratch alone (current API)", scratch_flags, true_flags),
        score_option("C  transformer OR scratch", transformer_flags | scratch_flags, true_flags, transformer_threshold),
        score_option("D  average of both", blended_scores >= blend_threshold, true_flags, blend_threshold),
    ]
    return options, transformer_threshold, blend_threshold


def export_run(trainer, tokenizer, transformer_threshold, blend_threshold, options):
    export_dir = RUN_DIR / "xlmr_urgent"
    export_dir.mkdir(parents=True, exist_ok=True)
    trainer.model.half().save_pretrained(str(export_dir))
    tokenizer.save_pretrained(str(export_dir))
    urgency_settings = {"threshold": transformer_threshold, "blend_threshold": blend_threshold, "input": "masked_nofiller"}
    (export_dir / "urgency.json").write_text(json.dumps(urgency_settings, indent=2))
    run_info = {
        "run_name": RUN_NAME,
        "initialized_from": "category fine-tuned xlm-roberta (xlmr_category/final)",
        "input_format": "[language] [channel] masked body with learned filler sentences removed",
        "hyperparameters": {"epochs": EPOCH_COUNT, "learning_rate": LEARNING_RATE, "batch_size": BATCH_SIZE, "max_tokens": MAX_TOKENS, "seed": 42},
        "validation_options": options,
        "versions": {"torch": torch.__version__, "transformers": transformers.__version__, "sklearn": sklearn.__version__},
    }
    (RUN_DIR / "run_info.json").write_text(json.dumps(run_info, indent=2))
    print("saved run to", RUN_DIR)


print("category model labels:", AutoModelForSequenceClassification.from_pretrained(str(CATEGORY_MODEL_DIR)).config.num_labels)
filler_sentences = joblib.load(SCRATCH_DIR / "filler_sentences.joblib")
scratch_urgent_model = joblib.load(SCRATCH_DIR / "is_urgent_model.joblib")
language_detector = joblib.load(DETECTOR_PATH)
detected_validation_languages = language_detector.predict(build_detector_inputs(validation_data))

train_texts = build_transformer_inputs(train_data, train_data["language"], filler_sentences)
validation_texts = build_transformer_inputs(validation_data, detected_validation_languages, filler_sentences)
print(train_texts[0][:80])

tokenizer = AutoTokenizer.from_pretrained(str(CATEGORY_MODEL_DIR))
model = load_model_from_category_checkpoint()
train_dataset = build_urgent_dataset(train_texts, train_data["is_urgent"], tokenizer)
validation_dataset = build_urgent_dataset(validation_texts, validation_data["is_urgent"], tokenizer)

trainer = Trainer(
    model=model,
    args=build_training_arguments(len(train_data)),
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)
trainer.train()

true_flags = validation_data["is_urgent"].to_numpy().astype(int)
transformer_probabilities = urgent_probabilities(trainer.predict(validation_dataset).predictions)
scratch_scores = np.asarray(scratch_urgent_model.decision_function(build_scratch_inputs(validation_data, filler_sentences)))
options, transformer_threshold, blend_threshold = compare_options(transformer_probabilities, scratch_scores, true_flags)
for option in options:
    print(json.dumps(option))
export_run(trainer, tokenizer, transformer_threshold, blend_threshold, options)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

category model labels: 11
[en] [chat] the fare was supposed to be <amount> but a different sum was collect


[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `11`.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/tensorforge/xlmr_category/final
Key                        | Status   |                                                                                        
---------------------------+----------+----------------------------------------------------------------------------------------
classifier.out_proj.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([11]) vs model:torch.Size([2])          
classifier.out_proj.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([11, 768]) vs model:torch.Size([2, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{"option": "A  transformer v2 alone", "urgent_f1": 0.9412, "precision": 0.9863, "recall": 0.9, "threshold": 0.13}
{"option": "B  scratch alone (current API)", "urgent_f1": 0.8947, "precision": 0.9444, "recall": 0.85, "threshold": null}
{"option": "C  transformer OR scratch", "urgent_f1": 0.9441, "precision": 0.9383, "recall": 0.95, "threshold": 0.13}
{"option": "D  average of both", "urgent_f1": 0.9487, "precision": 0.9737, "recall": 0.925, "threshold": 0.27}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved run to /content/drive/MyDrive/tensorforge/runs/urgent_v2_20261010_0525


In [18]:
import shutil
from google.colab import files

zip_path = shutil.make_archive("/content/xlmr_urgent_v2", "zip", RUN_DIR / "xlmr_urgent")
files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>